## Config

In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import os
import time

import joblib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.signal import savgol_filter
from sklearn.metrics import root_mean_squared_error

IS_KAGGLE = Path('/kaggle/input').exists()
OUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('./output_aw_postprocess_research_v2')
ARTIFACT_DIR = OUT_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_V1_OOF_RMSE = 10.4521227
BASELINE_V1_LB = 9.964
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WINNING_WEIGHTS = {
    'catboost-2': 0.2718483,
    'catboost-3': 0.3763824,
    'lightgbm-1': 0.3517693,
}
WINNING_PP_PARAMS = {
    'alpha': 1.04,
    'tau': 65,
    'w_pf': 0.07,
    'sg_w': 27,
    'sg_p': 3,
}
NOTEBOOK13_FALLBACK_SELECTION = {
    'source_notebook': '13_aw_submit_postprocess_research',
    'lb_score': BASELINE_V1_LB,
    'candidate_oof_rmse': BASELINE_V1_OOF_RMSE,
    'use_candidate': True,
    'weights': WINNING_WEIGHTS,
    'best_pp_params': WINNING_PP_PARAMS,
}

PREP_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv05/rogii-eda-features'),
    Path('./output_aw/model_artifacts'),
]
MODEL_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv051/rogii-train-lightgbm'),
    Path('/kaggle/input/notebooks/baopv051/rogii-train-catboost'),
    Path('./output_aw_lgbm/model_artifacts'),
    Path('./output_aw_catboost/model_artifacts'),
]
SELECTION_CANDIDATES = [
    Path('/kaggle/input/rogii-climbhill'),
    Path('/kaggle/input/rogii-climbhill/model_artifacts'),
    Path('/kaggle/input/rogii-aw-select/model_artifacts'),
    Path('./output_aw_select/model_artifacts'),
]
POSTPROCESS_V1_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv051/rogii-postprocess-research/model_artifacts'),
    Path('./output_aw_postprocess_research/model_artifacts'),
]

def first_existing(candidates, required):
    for p in candidates:
        if all((p / r).exists() for r in required):
            return p
    raise FileNotFoundError(f'No candidate contains {required}: {candidates}')

def optional_existing(candidates, required):
    for p in candidates:
        if all((p / r).exists() for r in required):
            return p
    return None

PREP_DIR = first_existing(PREP_CANDIDATES, ['train_df.joblib', 'features.json'])
MODEL_DIRS = [p for p in MODEL_CANDIDATES if (p / 'oof').exists()]
SELECTION_DIR = optional_existing(SELECTION_CANDIDATES, ['selection.json'])
POSTPROCESS_V1_DIR = optional_existing(POSTPROCESS_V1_CANDIDATES, ['selection_postprocess_research.json'])

print('IS_KAGGLE:', IS_KAGGLE)
print('PREP_DIR:', PREP_DIR)
print('MODEL_DIRS:', MODEL_DIRS)
print('SELECTION_DIR:', SELECTION_DIR)
print('POSTPROCESS_V1_DIR:', POSTPROCESS_V1_DIR)
print('ARTIFACT_DIR:', ARTIFACT_DIR)
if not MODEL_DIRS:
    raise RuntimeError('No model OOF directories found. Attach Train LightGBM and Train CatBoost outputs.')

## Load OOFs

In [ ]:
t0 = time.time()
train_df = joblib.load(PREP_DIR / 'train_df.joblib')
features = json.loads((PREP_DIR / 'features.json').read_text(encoding='utf-8'))
y_resid = train_df['target'].to_numpy(np.float32)
base = train_df['last_known_tvt'].to_numpy(np.float32)
y_abs = y_resid + base
pf_resid = (train_df['pf_ancc'].to_numpy(np.float32) - base).astype(np.float32)

oof = {}
model_scores = {}
for model_dir in MODEL_DIRS:
    for oof_file in sorted((model_dir / 'oof').glob('*_oof.npy')):
        name = oof_file.name.replace('_oof.npy', '')
        pred = np.load(oof_file).astype(np.float32)
        if len(pred) != len(train_df):
            print('skip length mismatch:', oof_file, len(pred), len(train_df))
            continue
        oof[name] = pred
        model_scores[name] = float(root_mean_squared_error(y_resid, pred))
        print(f'{name:14s} residual OOF RMSE: {model_scores[name]:.6f}')

if not oof:
    raise RuntimeError('No usable OOF arrays found.')
oof_df = pd.DataFrame(oof)
names = list(oof_df.columns)
print('train_df:', train_df.shape)
print('n_features:', len(features))
print('OOF candidates:', names)
print('load minutes:', round((time.time() - t0) / 60, 2))

previous_selection = None
if SELECTION_DIR is not None:
    previous_selection = json.loads((SELECTION_DIR / 'selection.json').read_text(encoding='utf-8'))
    print('previous hillclimb final_oof_rmse:', previous_selection.get('final_oof_rmse'))
    print('previous hillclimb weights:', previous_selection.get('weights'))

loaded_v1_selection = None
if POSTPROCESS_V1_DIR is not None:
    loaded_v1_selection = json.loads((POSTPROCESS_V1_DIR / 'selection_postprocess_research.json').read_text(encoding='utf-8'))
    print('loaded v1 postprocess selection:', POSTPROCESS_V1_DIR)
    print('v1 use_candidate:', loaded_v1_selection.get('use_candidate'))
    print('v1 candidate OOF:', loaded_v1_selection.get('candidate_final_oof_rmse', loaded_v1_selection.get('candidate_oof_rmse')))

notebook13_selection = loaded_v1_selection or NOTEBOOK13_FALLBACK_SELECTION

## Blend Candidates

Build candidate residual blends from the notebook 13 winner, prior hill-climb weights, constrained least-squares blends, and deterministic small perturbations around the winning weights.

In [ ]:
def score_residual(pred):
    return float(root_mean_squared_error(y_resid, np.asarray(pred, dtype=np.float32)))

def normalize_weights(weights):
    weights = {k: float(v) for k, v in weights.items() if k in oof and float(v) > 0}
    total = sum(weights.values())
    if total <= 0:
        return {}
    return {k: v / total for k, v in weights.items()}

def blend_from_weights(weights):
    weights = normalize_weights(weights)
    out = np.zeros(len(train_df), dtype=np.float32)
    for name, weight in weights.items():
        out += float(weight) * oof[name]
    return out, weights

def optimize_convex(candidate_names, label):
    candidate_names = list(candidate_names)
    mat = np.column_stack([oof[n] for n in candidate_names]).astype(np.float32)
    y = y_resid.astype(np.float32)
    cov = (mat.T @ mat).astype(np.float64) / len(y)
    rhs = (mat.T @ y).astype(np.float64) / len(y)
    n = len(candidate_names)
    x0 = np.full(n, 1.0 / n, dtype=np.float64)
    bounds = [(0.0, 1.0)] * n
    constraints = [{'type': 'eq', 'fun': lambda w: float(np.sum(w) - 1.0)}]

    def obj(w):
        w = np.asarray(w, dtype=np.float64)
        return float(w @ cov @ w - 2.0 * rhs @ w)

    result = minimize(obj, x0, method='SLSQP', bounds=bounds, constraints=constraints, options={'maxiter': 200, 'ftol': 1e-10})
    w = np.clip(result.x, 0, 1)
    w = w / w.sum()
    weights = {name: float(val) for name, val in zip(candidate_names, w) if val > 1e-6}
    pred, weights = blend_from_weights(weights)
    print(label, 'success=', result.success, 'rmse=', score_residual(pred), 'weights=', weights)
    del mat
    gc.collect()
    return pred, weights

def add_blend(label, weights, family):
    pred, weights = blend_from_weights(weights)
    if not weights:
        print('skip blend with no available weights:', label, weights)
        return
    blend_candidates[label] = pred
    blend_weights[label] = weights
    blend_family[label] = family

def perturb_winner_weights():
    base_weights = normalize_weights(WINNING_WEIGHTS)
    keys = [k for k in WINNING_WEIGHTS if k in base_weights]
    if len(keys) < 2:
        return {}

    variants = {}
    for eps in [0.01, 0.025, 0.04]:
        for src in keys:
            for dst in keys:
                if src == dst:
                    continue
                w = dict(base_weights)
                move = min(eps, w[src] * 0.5)
                w[src] -= move
                w[dst] += move
                variants[f'winner_shift_{src}_to_{dst}_{eps:.3f}'] = normalize_weights(w)

    rng = np.random.default_rng(RANDOM_SEED)
    base_vec = np.array([base_weights[k] for k in keys], dtype=np.float64)
    for scale in [0.015, 0.03]:
        for i in range(4):
            noise = rng.normal(0.0, scale, size=len(keys))
            vec = np.clip(base_vec + noise, 1e-6, None)
            vec = vec / vec.sum()
            variants[f'winner_jitter_s{scale:.3f}_{i:02d}'] = {k: float(v) for k, v in zip(keys, vec)}
    return variants

blend_candidates = {}
blend_weights = {}
blend_family = {}

best_single = min(model_scores, key=model_scores.get)
add_blend(f'single_{best_single}', {best_single: 1.0}, 'single')
add_blend('notebook13_winner', WINNING_WEIGHTS, 'v1_winner')

if previous_selection and previous_selection.get('weights'):
    add_blend('previous_hc', previous_selection['weights'], 'previous_hillclimb')

top3 = [name for name, _ in sorted(model_scores.items(), key=lambda kv: kv[1])[:3]]
pred, weights = optimize_convex(top3, 'slsqp_top3')
add_blend('slsqp_top3', weights, 'slsqp')

pred, weights = optimize_convex(names, 'slsqp_all')
add_blend('slsqp_all', weights, 'slsqp')

for label, weights in perturb_winner_weights().items():
    add_blend(label, weights, 'winner_perturbation')

for label, pred in blend_candidates.items():
    print(f'{label:32s} residual RMSE: {score_residual(pred):.6f}', blend_weights[label])

print('blend candidate count:', len(blend_candidates))

## Postprocess Search

Search only the local neighborhood around notebook 13: `alpha=1.00..1.08`, `tau=45..95`, `w_pf=0.04..0.11`, and odd Savitzky-Golay windows from 19 to 39.

In [ ]:
alpha_grid = [round(x, 2) for x in np.arange(1.00, 1.0801, 0.01)]
tau_grid = list(range(45, 100, 5))
w_pf_grid = [round(x, 2) for x in np.arange(0.04, 0.1101, 0.01)]
smooth_windows = list(range(19, 40, 2))
smooth_top_n = 80

md_since_arr = np.maximum(train_df['md_since'].to_numpy(np.float32), 0.0)
ramp_by_tau = {int(tau): (1.0 - np.exp(-md_since_arr / float(tau))).astype(np.float32) for tau in tau_grid}
well_indices = [np.asarray(idx) for _, idx in pd.DataFrame({'well': train_df['well'].to_numpy()}).groupby('well', sort=False).indices.items()]

def rmse_abs(pred_abs):
    diff = pred_abs.astype(np.float32) - y_abs
    return float(np.sqrt(np.mean(diff * diff, dtype=np.float64)))

def apply_pp(model_resid, alpha, tau, w_pf):
    d = model_resid * (1.0 - w_pf) + pf_resid * w_pf
    ramp = ramp_by_tau.get(int(tau))
    if ramp is not None:
        d = d * ramp
    return (d * float(alpha)).astype(np.float32)

def sg_smooth_abs(abs_pred, sg_w=17, sg_p=3):
    if not sg_w or sg_w <= 1:
        return abs_pred.astype(np.float32)
    out = abs_pred.astype(np.float32).copy()
    for loc in well_indices:
        vals = out[loc]
        wl = min(int(sg_w), len(vals))
        if wl % 2 == 0:
            wl -= 1
        if wl >= sg_p + 2:
            out[loc] = savgol_filter(vals, wl, sg_p).astype(np.float32)
    return out

stage1_rows = []
t0 = time.time()
total_eval = len(blend_candidates) * len(alpha_grid) * len(tau_grid) * len(w_pf_grid)
done_eval = 0
print('stage1 grid evaluations:', total_eval)
for blend_i, (blend_name, model_resid) in enumerate(blend_candidates.items(), start=1):
    blend_t0 = time.time()
    best_for_blend = float('inf')
    for alpha in alpha_grid:
        for tau in tau_grid:
            ramp = ramp_by_tau[int(tau)]
            for w_pf in w_pf_grid:
                resid = ((model_resid * (1.0 - w_pf) + pf_resid * w_pf) * ramp * float(alpha)).astype(np.float32)
                score = rmse_abs(base + resid)
                if score < best_for_blend:
                    best_for_blend = score
                stage1_rows.append({
                    'stage': 'raw',
                    'blend': blend_name,
                    'blend_family': blend_family[blend_name],
                    'alpha': float(alpha),
                    'tau': int(tau),
                    'w_pf': float(w_pf),
                    'sg_w': 0,
                    'sg_p': 3,
                    'rmse': score,
                })
                done_eval += 1
    elapsed = time.time() - t0
    print(
        f'stage1 blend {blend_i}/{len(blend_candidates)} {blend_name}: '
        f'best={best_for_blend:.6f}, elapsed={elapsed/60:.1f}m, '
        f'rate={done_eval/max(elapsed, 1e-9):.1f} eval/s'
    )
stage1 = pd.DataFrame(stage1_rows).sort_values('rmse').reset_index(drop=True)
print('stage1 best:')
display(stage1.head(20))
print('stage1 minutes:', round((time.time() - t0) / 60, 2))

stage2_rows = []
t0 = time.time()
stage2_source = stage1.head(smooth_top_n).to_dict('records')
print('stage2 smoothing rows:', len(stage2_source), 'windows:', smooth_windows)
for row_i, row in enumerate(stage2_source, start=1):
    model_resid = blend_candidates[row['blend']]
    resid = apply_pp(model_resid, row['alpha'], row['tau'], row['w_pf'])
    raw_abs = base + resid
    best_for_row = float('inf')
    for sg_w in smooth_windows:
        pred_abs = sg_smooth_abs(raw_abs, sg_w=sg_w, sg_p=3)
        score = rmse_abs(pred_abs)
        if score < best_for_row:
            best_for_row = score
        out = dict(row)
        out['stage'] = 'smooth'
        out['sg_w'] = int(sg_w)
        out['rmse'] = score
        stage2_rows.append(out)
    if row_i == 1 or row_i % 10 == 0 or row_i == len(stage2_source):
        print(f'stage2 row {row_i}/{len(stage2_source)} best={best_for_row:.6f}, elapsed={(time.time() - t0)/60:.1f}m')
stage2 = pd.DataFrame(stage2_rows).sort_values('rmse').reset_index(drop=True)
print('stage2 best:')
display(stage2.head(30))
print('stage2 minutes:', round((time.time() - t0) / 60, 2))

results = pd.concat([stage1, stage2], ignore_index=True).sort_values('rmse').reset_index(drop=True)
results.to_csv(ARTIFACT_DIR / 'postprocess_research_v2_results.csv', index=False)
best = results.iloc[0].to_dict()
print('best:', best)
print('baseline v1 OOF:', BASELINE_V1_OOF_RMSE)
print('improvement vs baseline:', float(BASELINE_V1_OOF_RMSE - best['rmse']))

## Save Candidate Selection

In [ ]:
best_blend = str(best['blend'])
candidate_oof = float(best['rmse'])
selection = {
    'research_notebook': '14_aw_postprocess_research_v2',
    'baseline_v1_oof_rmse': float(BASELINE_V1_OOF_RMSE),
    'baseline_v1_lb': float(BASELINE_V1_LB),
    'candidate_oof_rmse': candidate_oof,
    'improvement': float(BASELINE_V1_OOF_RMSE - candidate_oof),
    'use_candidate': bool(candidate_oof + 1e-9 < BASELINE_V1_OOF_RMSE),
    'blend_name': best_blend,
    'blend_family': str(best.get('blend_family', blend_family.get(best_blend, 'unknown'))),
    'weights': {str(k): float(v) for k, v in blend_weights[best_blend].items()},
    'best_pp_params': {
        'alpha': float(best['alpha']),
        'tau': int(best['tau']),
        'w_pf': float(best['w_pf']),
        'sg_w': int(best['sg_w']),
        'sg_p': int(best.get('sg_p', 3)),
    },
    'guardrail': {
        'candidate_must_beat_oof': float(BASELINE_V1_OOF_RMSE),
        'create_submission_notebook_only_if_use_candidate': True,
    },
    'search_space': {
        'alpha_grid': [float(x) for x in alpha_grid],
        'tau_grid': [int(x) for x in tau_grid],
        'w_pf_grid': [float(x) for x in w_pf_grid],
        'smooth_windows': [int(x) for x in smooth_windows],
        'smooth_top_n': int(smooth_top_n),
        'blend_count': int(len(blend_candidates)),
    },
    'model_scores': {str(k): float(v) for k, v in model_scores.items()},
    'previous_hillclimb_selection': previous_selection,
    'previous_notebook13_selection': notebook13_selection,
}
(ARTIFACT_DIR / 'selection_postprocess_research_v2.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
print(json.dumps(selection, indent=2)[:5000])
print('saved files:')
for p in sorted(ARTIFACT_DIR.glob('*')):
    print(p.name, round(p.stat().st_size / 1024**2, 3), 'MB')

if selection['use_candidate']:
    print('Candidate improved OOF. Next step: create 15_aw_submit_postprocess_research_v2.ipynb from notebook 13 and selection_postprocess_research_v2.json.')
else:
    print('No OOF improvement. Keep notebook 13 as the current best submission and move to a separate CPU feature experiment.')